# SSIF_V3 模型訓練 Notebook（繁體中文）

本 Notebook 從已完成轉換的 `training_archive_json` 開始，依序完成：資料驗證、事件層級四集合切分、EW10 快速測試、EW10–EW40 正式訓練、結果彙整與 checkpoint 稽核。

科學隔離原則：**validation 選最佳 epoch；calibration 選 alert threshold；test 僅在兩者固定後評估。**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 安全同步 repository

In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO_ROOT = Path('/content/SSIF_V3')
REPO_URL = 'https://github.com/oceanicdayi/SSIF_V3.git'

def run_checked(command, cwd=None, capture=False):
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=capture)
    if capture:
        if result.stdout: print(result.stdout, end='')
        if result.stderr: print(result.stderr, end='')
    if result.returncode:
        raise RuntimeError(f"exit {result.returncode}: " + " ".join(map(str, command)))
    return result

os.chdir('/content')
if (REPO_ROOT / '.git').is_dir():
    try:
        run_checked(['git','-C',str(REPO_ROOT),'fetch','--prune','origin'])
        run_checked(['git','-C',str(REPO_ROOT),'reset','--hard','origin/main'])
        run_checked(['git','-C',str(REPO_ROOT),'clean','-fd'])
    except RuntimeError:
        os.chdir('/content')
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        run_checked(['git','clone','--depth','1',REPO_URL,str(REPO_ROOT)])
else:
    os.chdir('/content')
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    run_checked(['git','clone','--depth','1',REPO_URL,str(REPO_ROOT)])

REPO_SHA = run_checked(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'], capture=True).stdout.strip()
print('Repository commit:', REPO_SHA)
run_checked(['python','-m','pip','install','-q','-r',str(REPO_ROOT/'requirements.txt')])

## 2. 路徑與訓練設定

In [ ]:
from datetime import datetime
import json, math, platform, random, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')
TRAIN_DATA = WORK_ROOT / 'data' / 'training_archive_json'
EXTERNAL_DATA = WORK_ROOT / 'data' / 'external_evaluation_json'
PREPARED_DIR = WORK_ROOT / 'prepared' / 'split_v1'
SPLIT_MANIFEST = PREPARED_DIR / 'split_manifest.json'
QUICK_MODEL_DIR = WORK_ROOT / 'models' / 'quick_EW10'
FULL_MODEL_DIR = WORK_ROOT / 'models' / 'ssif_v3_seed_20260728'
REPORT_DIR = WORK_ROOT / 'reports' / 'training_seed_20260728'
EXTERNAL_OUTPUT_DIR = WORK_ROOT / 'inference' / 'external_seed_20260728'

WINDOWS = [10,15,20,25,30,35,40]
SEED = 20260728
LABEL_HORIZON = 120
MIN_VALID = 0.80
MIN_PRECISION = 0.90
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
WORKERS = 2

RUN_DATA_VALIDATION = True
CREATE_SPLIT_IF_MISSING = True
REBUILD_SPLIT = False
RUN_QUICK_TRAIN = True
RUN_FULL_TRAIN = False
RUN_EXTERNAL_EVALUATION = False
OVERWRITE_QUICK_MODEL = True
OVERWRITE_FULL_MODEL = False

for p in [PREPARED_DIR, QUICK_MODEL_DIR.parent, REPORT_DIR, EXTERNAL_OUTPUT_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)
assert TRAIN_DATA.is_dir()
EVENT_FILES = sorted(TRAIN_DATA.glob('event_*.json'))
assert EVENT_FILES, f'找不到 event_*.json：{TRAIN_DATA}'
print('Events:', len(EVENT_FILES))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PyTorch:', torch.__version__)

## 3. 資料驗證與環境紀錄

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

REPORT_DIR.mkdir(parents=True, exist_ok=True)
environment = {
    'created_at_utc': datetime.utcnow().isoformat(timespec='seconds')+'Z',
    'repository_commit': REPO_SHA,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seed': SEED,
}
(REPORT_DIR/'environment.json').write_text(json.dumps(environment,ensure_ascii=False,indent=2),encoding='utf-8')

if RUN_DATA_VALIDATION:
    validation_path = REPORT_DIR/'archive_validation.json'
    run_checked([
        'python','combined_csv_to_ssif_json.py','validate',
        '--data-dir',str(TRAIN_DATA),'--horizon',str(LABEL_HORIZON),
        '--max-errors','100','--report',str(validation_path)
    ], cwd=REPO_ROOT)
    validation = json.loads(validation_path.read_text(encoding='utf-8'))
    display(pd.DataFrame.from_dict(validation['counters'],orient='index',columns=['count']))
    assert validation['counters'].get('errors',0) == 0
    assert validation['counters'].get('event_json',0) == len(EVENT_FILES)
    print('PASS: archive validation')

## 4. 建立或載入固定事件層級 split

In [ ]:
if REBUILD_SPLIT and PREPARED_DIR.exists():
    shutil.rmtree(PREPARED_DIR)
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)

if not SPLIT_MANIFEST.is_file():
    assert CREATE_SPLIT_IF_MISSING
    run_checked([
        'python','prepare_ssif_dataset.py','audit-split',
        '--data-dir',str(TRAIN_DATA),'--output-dir',str(PREPARED_DIR),
        '--windows',*map(str,WINDOWS),'--label-horizon',str(LABEL_HORIZON),
        '--min-label-valid-fraction',str(MIN_VALID),
        '--min-window-valid-fraction',str(MIN_VALID),
        '--train-ratio','0.70','--validation-ratio','0.10',
        '--calibration-ratio','0.10','--test-ratio','0.10',
        '--split-candidates','5000','--seed',str(SEED)
    ], cwd=REPO_ROOT)

manifest = json.loads(SPLIT_MANIFEST.read_text(encoding='utf-8'))
assert manifest['validation']['valid']
assert manifest['windows'] == WINDOWS
assert manifest['label_horizon'] == LABEL_HORIZON
counts = {k:len(v) for k,v in manifest['splits'].items()}
print('Fingerprint:', manifest['data_fingerprint_sha256'])
display(pd.DataFrame([{'split':k,'events':v} for k,v in counts.items()]))

audit = json.loads((PREPARED_DIR/'audit_summary.json').read_text(encoding='utf-8'))
assert audit['n_duplicate_event_ids'] == 0
split_df = pd.read_csv(PREPARED_DIR/'event_split.csv')
display(split_df.groupby('split').agg(events=('event_id','nunique'),records=('n_station_records','sum'),positive_rate=('has_event_positive','mean'),median_mag=('magnitude','median')).reset_index())

## 5. 訓練命令

In [ ]:
def train_command(output_dir, windows, epochs):
    command = [
        'python','train_ssif_v3.py','train-all',
        '--data-dir',str(TRAIN_DATA),'--split-manifest',str(SPLIT_MANIFEST),
        '--output-dir',str(output_dir),'--windows',*map(str,windows),
        '--label-horizon',str(LABEL_HORIZON),'--cohort','common',
        '--epochs',str(epochs),'--batch-size',str(BATCH_SIZE),
        '--eval-batch-size',str(EVAL_BATCH_SIZE),'--lr','3e-4',
        '--weight-decay','1e-2','--warmup-ratio','0.10',
        '--min-precision',str(MIN_PRECISION),'--hidden-size','192',
        '--num-layers','4','--num-heads','4','--ff-mult','2',
        '--dropout','0.1','--conv1','96','--conv2','192',
        '--loss-cls','0.45','--loss-alert','0.35',
        '--loss-ordinal','0.15','--loss-consistency','0.05',
        '--seed',str(SEED),'--window-seed-mode','same',
        '--patience','6','--workers',str(WORKERS)
    ]
    if torch.cuda.is_available(): command.append('--amp')
    return command

## 6. EW10 一個 epoch 快速測試

In [ ]:
if RUN_QUICK_TRAIN:
    if QUICK_MODEL_DIR.exists() and any(QUICK_MODEL_DIR.iterdir()):
        assert OVERWRITE_QUICK_MODEL
        shutil.rmtree(QUICK_MODEL_DIR)
    run_checked(train_command(QUICK_MODEL_DIR,[10],1), cwd=REPO_ROOT)
    assert (QUICK_MODEL_DIR/'EW10'/'best.pt').is_file()
    quick = json.loads((QUICK_MODEL_DIR/'summary.json').read_text(encoding='utf-8'))[0]
    display(pd.DataFrame([{
        'window':quick['window'],'best_epoch':quick['best_epoch'],
        'threshold':quick['threshold'],'precision':quick['test']['alert']['precision'],
        'pod':quick['test']['alert']['pod'],'f1':quick['test']['alert']['f1'],
        'fpr':quick['test']['alert']['fpr']
    }]))
    print('PASS: EW10 quick training')
else:
    print('RUN_QUICK_TRAIN=False')

## 7. 正式訓練 EW10–EW40

In [ ]:
if RUN_FULL_TRAIN:
    if FULL_MODEL_DIR.exists() and any(FULL_MODEL_DIR.iterdir()):
        assert OVERWRITE_FULL_MODEL, '正式模型已存在；請改 run 名稱或明確允許覆蓋'
        shutil.rmtree(FULL_MODEL_DIR)
    run_checked(train_command(FULL_MODEL_DIR,WINDOWS,30), cwd=REPO_ROOT)
    assert (FULL_MODEL_DIR/'summary.json').is_file()
    for w in WINDOWS:
        run_dir = FULL_MODEL_DIR/f'EW{w:02d}'
        assert (run_dir/'best.pt').is_file()
        assert (run_dir/'history.json').is_file()
        assert (run_dir/'metrics.json').is_file()
    print('PASS: full training')
else:
    print('RUN_FULL_TRAIN=False；quick test 通過後再開啟')

## 8. 彙整正式模型與繪圖

In [ ]:
def result_table(model_dir):
    path = model_dir/'summary.json'
    if not path.is_file(): return pd.DataFrame()
    rows=[]
    for x in json.loads(path.read_text(encoding='utf-8')):
        a=x['test']['alert']
        rows.append({'window':x['window'],'best_epoch':x['best_epoch'],'threshold':x['threshold'],'precision':a['precision'],'pod':a['pod'],'f1':a['f1'],'fpr':a['fpr'],'n':x['test']['n_samples']})
    return pd.DataFrame(rows).sort_values('window')

results = result_table(FULL_MODEL_DIR)
if results.empty:
    print('尚無正式 summary.json')
else:
    display(results)
    results.to_csv(REPORT_DIR/'model_performance_by_window.csv',index=False,encoding='utf-8-sig')
    plt.figure(figsize=(9,5))
    for col in ['precision','pod','f1']:
        plt.plot(results['window'],results[col],marker='o',label=col.upper())
    plt.xlabel('Early window (s)'); plt.ylabel('Score'); plt.ylim(0,1.02)
    plt.xticks(WINDOWS); plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
    plt.savefig(REPORT_DIR/'test_metrics_by_window.png',dpi=180); plt.show()

## 9. Checkpoint 與資料指紋稽核

In [ ]:
rows=[]
for w in WINDOWS:
    path=FULL_MODEL_DIR/f'EW{w:02d}'/'best.pt'
    if not path.is_file():
        rows.append({'window':w,'exists':False}); continue
    payload=torch.load(path,map_location='cpu',weights_only=False)
    meta=payload.get('training_metadata',{})
    rows.append({'window':w,'exists':True,'checkpoint_window':payload.get('window'),'best_epoch':meta.get('best_epoch'),'threshold':payload.get('alert_probability_threshold'),'fingerprint_matches':meta.get('data_fingerprint_sha256')==manifest['data_fingerprint_sha256'],'label_horizon':meta.get('label_horizon'),'cohort':meta.get('cohort')})
checkpoint_audit=pd.DataFrame(rows)
display(checkpoint_audit)
if RUN_FULL_TRAIN:
    assert checkpoint_audit['exists'].all()
    assert checkpoint_audit['fingerprint_matches'].all()
    print('PASS: checkpoints match frozen data fingerprint')

## 10. 選擇性：完全獨立 archive inference

In [ ]:
if RUN_EXTERNAL_EVALUATION:
    assert list(EXTERNAL_DATA.glob('event_*.json')), '找不到獨立 evaluation archive'
    assert (FULL_MODEL_DIR/'summary.json').is_file()
    if EXTERNAL_OUTPUT_DIR.exists() and any(EXTERNAL_OUTPUT_DIR.iterdir()):
        raise RuntimeError('外部評估輸出已存在；請使用新的輸出目錄')
    run_checked([
        'python','train_ssif_v3.py','evaluate-all',
        '--data-dir',str(EXTERNAL_DATA),'--model-root',str(FULL_MODEL_DIR),
        '--output-dir',str(EXTERNAL_OUTPUT_DIR),'--windows',*map(str,WINDOWS),
        '--label-horizon',str(LABEL_HORIZON),'--cohort','common',
        '--batch-size','128','--workers',str(WORKERS)
    ], cwd=REPO_ROOT)
else:
    print('RUN_EXTERNAL_EVALUATION=False')

## 11. 保存 run inventory

In [ ]:
inventory={
    'repository_commit':REPO_SHA,
    'data_root':str(TRAIN_DATA),
    'data_fingerprint_sha256':manifest['data_fingerprint_sha256'],
    'split_manifest':str(SPLIT_MANIFEST),
    'quick_model_dir':str(QUICK_MODEL_DIR),
    'full_model_dir':str(FULL_MODEL_DIR),
    'windows':WINDOWS,'seed':SEED,'label_horizon':LABEL_HORIZON,
    'min_valid_fraction':MIN_VALID,'min_precision':MIN_PRECISION,
    'flags':{'quick':RUN_QUICK_TRAIN,'full':RUN_FULL_TRAIN,'external':RUN_EXTERNAL_EVALUATION}
}
(REPORT_DIR/'run_inventory.json').write_text(json.dumps(inventory,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(inventory,ensure_ascii=False,indent=2))

## 執行順序

1. 先執行到第 6 節，確認資料、split 與 EW10 quick training 都通過。  
2. 保持同一份 `split_manifest.json`，不要根據模型結果重切資料。  
3. 將 `RUN_FULL_TRAIN=True` 後執行第 7 節。  
4. 執行第 8–9 節，保存表格、圖與 checkpoint 指紋稽核。  
5. 只有具備完全獨立事件 archive 時才執行第 10 節。